# 这里用https://transformers.run/的例子

In [1]:
from torch import nn
from transformers import AutoConfig
from transformers import AutoTokenizer

model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

text = "time flies like an arrow"
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
print(inputs.input_ids)

config = AutoConfig.from_pretrained(model_ckpt)
token_emb = nn.Embedding(config.vocab_size, config.hidden_size)
print(token_emb)

inputs_embeds = token_emb(inputs.input_ids)
print(inputs_embeds.size())

tensor([[ 2051, 10029,  2066,  2019,  8612]])
Embedding(30522, 768)
torch.Size([1, 5, 768])


In [2]:
import torch
from math import sqrt

Q = K = V = inputs_embeds
dim_k = K.size(-1)
scores = torch.bmm(Q, K.transpose(1,2)) / sqrt(dim_k)
print(scores.size())

torch.Size([1, 5, 5])


In [3]:
import torch.nn.functional as F

weights = F.softmax(scores, dim=-1)
print(weights)
print(weights.sum(dim=-1))

tensor([[[1.0000e+00, 8.8053e-13, 8.1616e-14, 4.1557e-13, 6.2185e-13],
         [2.8505e-12, 1.0000e+00, 2.1388e-12, 1.5544e-12, 5.1907e-13],
         [7.1349e-14, 5.7757e-13, 1.0000e+00, 1.7446e-13, 5.8534e-13],
         [2.5595e-13, 2.9572e-13, 1.2291e-13, 1.0000e+00, 1.6641e-13],
         [2.7709e-11, 7.1446e-12, 2.9835e-11, 1.2039e-11, 1.0000e+00]]],
       grad_fn=<SoftmaxBackward0>)
tensor([[1., 1., 1., 1., 1.]], grad_fn=<SumBackward1>)


In [4]:
attn_outputs = torch.bmm(weights, V)
print(attn_outputs.shape)

torch.Size([1, 5, 768])
